In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_medication_prescription;
SET search_path TO qdp_medication_prescription;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.medication_prescription',
  '0.1',
  'medication_prescription.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_medication_prescription table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_medication_prescription CASCADE;

CREATE TABLE hub_medication_prescription (
    medication_prescription_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    medication_prescription_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_medication_prescription ADD CONSTRAINT pk_medprescription_hubmedprescription_medprescriptionid PRIMARY KEY (medication_prescription_id);

COMMENT ON COLUMN hub_medication_prescription.medication_prescription_id IS 'Medication Prescription Identity';




-- -----------------------------------------------------------
-- create sat_medication_prescription table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_prescription;

CREATE TABLE sat_medication_prescription (
    sat_medication_prescription_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    medication_prescription_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	encounter_id BIGINT,
	party_id BIGINT,
	prescription_ref VARCHAR,
	prior_prescription_ref VARCHAR,
	record_id BIGINT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_prescription ADD CONSTRAINT pk_medprescription_satmedprescription_satmedprescriptionid PRIMARY KEY (sat_medication_prescription_id);
ALTER TABLE sat_medication_prescription ADD CONSTRAINT fk_medprescription_satmedprescription_hubmedprescription_medprescriptionid FOREIGN KEY (medication_prescription_id) REFERENCES hub_medication_prescription(medication_prescription_id);

COMMENT ON COLUMN sat_medication_prescription.medication_prescription_id IS 'Medication Prescription Identity';
COMMENT ON COLUMN sat_medication_prescription.sat_medication_prescription_id IS 'Medication Prescription Identity Sat';
COMMENT ON COLUMN sat_medication_prescription.encounter_id IS 'The unique Patient Encounter Identifier for this Prescription. See https://build.fhir.org/medicationrequest-definitions.html#MedicationRequest.encounter';
COMMENT ON COLUMN sat_medication_prescription.party_id IS 'The Patient Identifier';
COMMENT ON COLUMN sat_medication_prescription.prescription_ref IS 'The GUID Reference Identifier for the Issued Prescription';
COMMENT ON COLUMN sat_medication_prescription.prior_prescription_ref IS 'Reference to an order/prescription that is being replaced by this Prescription';
COMMENT ON COLUMN sat_medication_prescription.record_id IS 'A Unique Identifier for a Data Record within the System that allows Data Ownership, Ownership History, Dynamic Attributes and Metadata to be associated with that Record';




-- -----------------------------------------------------------
-- create view_medication_prescriptions view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_medication_prescriptions AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.medication_prescription_id,
	h.medication_prescription_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_medication_prescription_id,
  s.encounter_id,
  s.party_id,
  s.prescription_ref,
  s.prior_prescription_ref,
  s.record_id,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_prescription.hub_medication_prescription h
    JOIN budd.qdp_medication_prescription.sat_medication_prescription s
      ON h.medication_prescription_id = s.medication_prescription_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_medication_prescriptions;


/**

**/
